In [ ]:
import numpy as np
import pandas as pd
import warnings

from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBRFClassifier
data = pd.read_csv("Clean_data.csv", na_values='?')
data = data.dropna()
data = data.apply(pd.to_numeric, errors='ignore')

X = data.drop(columns=["Depression"])
y = data["Depression"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

class SamaanEnsemble:
    def __init__(self):
        self.models = {
            "lr": LogisticRegression(max_iter=2000),
            "rf": RandomForestClassifier(n_estimators=100),
            "knn": KNeighborsClassifier(n_neighbors=5),
            "xgb": XGBRFClassifier()
        }
        self.router = DecisionTreeClassifier(max_depth=3)

    def fit(self, X, y):
        X_base, X_router, y_base, y_router = train_test_split(
            X, y, test_size=0.3, random_state=42
        )

        for model in self.models.values():
            model.fit(X_base, y_base)

        preds = []
        for model in self.models.values():
            preds.append(model.predict(X_router))

        preds = np.array(preds).T
        router_labels = []

        for i in range(len(y_router)):
            correct_models = []
            for j in range(len(self.models)):
                if preds[i][j] == y_router.iloc[i]:
                    correct_models.append(j)

            router_labels.append(correct_models[0] if len(correct_models) > 0 else 0)

        self.router.fit(X_router, np.array(router_labels))

    def predict(self, X):
        chosen_models = self.router.predict(X)
        final_preds = []
        model_keys = list(self.models.keys())

        for i in range(len(X)):
            model_idx = chosen_models[i]
            model_name = model_keys[model_idx]
            model = self.models[model_name]

            row = X.iloc[[i]]
            pred = model.predict(row)[0]
            final_preds.append(pred)
        return np.array(final_preds)

    def predict_proba(self, X):
        chosen_models = self.router.predict(X)
        final_probas = []
        model_keys = list(self.models.keys())

        for i in range(len(X)):
            model_idx = chosen_models[i]
            model_name = model_keys[model_idx]
            model = self.models[model_name]

            row = X.iloc[[i]]
            # Assuming the models have predict_proba and return probabilities for each class
            proba = model.predict_proba(row)[0]
            final_probas.append(proba)
        return np.array(final_probas)

sre = SamaanEnsemble()
sre.fit(X_train, y_train)
y_pred = sre.predict(X_test)
y_pred_proba = sre.predict_proba(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

/tmp/ipykernel_7521/969197418.py:19: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  data = data.apply(pd.to_numeric, errors='ignore')


Accuracy: 0.842652329749104

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.78      0.80      2313
           1       0.85      0.89      0.87      3267

    accuracy                           0.84      5580
   macro avg       0.84      0.83      0.84      5580
weighted avg       0.84      0.84      0.84      5580



In [ ]:
print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_proba[:, 1]))

ROC-AUC Score: 0.9142685220584841


In [ ]:
import pickle

# Save the model to a file
with open('samaan_ensemble_model.pkl', 'wb') as f:
    pickle.dump(sre, f)

print("Model successfully pickled to 'samaan_ensemble_model.pkl'")

Model successfully pickled to 'samaan_ensemble_model.pkl'
